# Tugas 3 — Visualisasi Transaksi POS Tahun 2025

Empat grafik dibuat dari dataset transaksi POS menggunakan Matplotlib.

## Persiapan Data

Data duplikat dihapus, format harga dibersihkan, dan tanggal diubah ke tipe waktu. Analisis penjualan memakai transaksi `SELESAI` dengan jumlah 1–12 unit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data").exists() and (ROOT_DIR.parent / "data").exists():
    ROOT_DIR = ROOT_DIR.parent

DATA_PATH = ROOT_DIR / "data" / "paket_a_transaksi_pos_2024_2025.csv"
OUTPUT_DIR = ROOT_DIR / "outputs" / "visualisasi" / "individual"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(DATA_PATH, dtype=str)
data = data.drop_duplicates().copy()

def bersihkan_harga(nilai):
    nilai = str(nilai).strip()
    if "," in nilai:
        nilai = nilai.replace(".", "").replace(",", ".")
    else:
        nilai = nilai.replace(".", "")
    return float(nilai)

data["harga_satuan"] = data["harga_satuan"].map(bersihkan_harga)
data["jumlah"] = data["jumlah"].astype(int)
data["diskon_persen"] = data["diskon_persen"].astype(int)
data["tanggal"] = pd.to_datetime(
    data["tanggal"], format="mixed", dayfirst=True
)
data["tahun"] = data["tanggal"].dt.year

data_2025 = data[data["tahun"] == 2025].copy()
data_jual = data_2025[
    (data_2025["status"] == "SELESAI")
    & (data_2025["jumlah"].between(1, 12))
].copy()

data_jual["Penjualan_Bersih"] = (
    data_jual["harga_satuan"]
    * data_jual["jumlah"]
    * (1 - data_jual["diskon_persen"] / 100)
)

print("Jumlah data setelah menghapus duplikat:", len(data))
print("Transaksi tahun 2025:", len(data_2025))
print("Transaksi SELESAI dengan jumlah 1–12 unit:", len(data_jual))

## 1. Penjualan Bersih per Bulan

Penjualan dihitung setelah diskon dan sebelum PPN.

In [ ]:
penjualan_bulanan = (
    data_jual.groupby(data_jual["tanggal"].dt.month)["Penjualan_Bersih"]
    .sum()
    .reindex(range(1, 13), fill_value=0)
)
nama_bulan = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun",
              "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

plt.figure(figsize=(10, 5))
plt.bar(nama_bulan, penjualan_bulanan.values, color="#3569A8")
plt.title("Penjualan Bersih per Bulan — 2025")
plt.xlabel("Bulan")
plt.ylabel("Penjualan Bersih (Rp)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "visualisasi_penjualan_bulanan_2025.png", dpi=160)
plt.show()

## 2. Sepuluh Produk Terlaris

Produk diurutkan berdasarkan total unit terjual pada transaksi selesai.

In [ ]:
produk_terlaris = (
    data_jual.groupby("kode_produk")["jumlah"]
    .sum()
    .nlargest(10)
    .sort_values()
)

plt.figure(figsize=(9, 6))
plt.barh(produk_terlaris.index, produk_terlaris.values, color="#4D8C78")
plt.title("10 Produk Terlaris — 2025")
plt.xlabel("Total Unit Terjual")
plt.ylabel("Kode Produk")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "visualisasi_10_produk_terlaris_2025.png", dpi=160)
plt.show()

## 3. Proporsi Status Transaksi

Status `SELESAI` dibandingkan dengan gabungan `BATAL` dan `RETUR`. Jumlah kedua status tersebut tetap ditampilkan terpisah.

In [ ]:
jumlah_status = data_2025["status"].value_counts()
berhasil = jumlah_status.get("SELESAI", 0)
tidak_selesai = jumlah_status.get("BATAL", 0) + jumlah_status.get("RETUR", 0)

plt.figure(figsize=(8, 5))
plt.pie(
    [berhasil, tidak_selesai],
    labels=["Berhasil\n(SELESAI)", "Tidak selesai\n(BATAL + RETUR)"],
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
    colors=["#4D8C78", "#D49A48"],
    wedgeprops={"width": 0.42, "edgecolor": "white"},
    pctdistance=0.78
)
plt.text(
    0, 0.05, f"{berhasil + tidak_selesai:,}".replace(",", "."),
    ha="center", va="center", fontsize=22, weight="bold"
)
plt.text(0, -0.16, "transaksi", ha="center", va="center", fontsize=9)
plt.title("Proporsi Status Transaksi — 2025")
plt.text(
    0, -1.2,
    f"BATAL: {jumlah_status.get('BATAL', 0)}  |  RETUR: {jumlah_status.get('RETUR', 0)}",
    ha="center"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "visualisasi_status_transaksi_2025.png", dpi=160)
plt.show()

print("SELESAI:", berhasil)
print("BATAL:", jumlah_status.get("BATAL", 0))
print("RETUR:", jumlah_status.get("RETUR", 0))

## 4. Hubungan Jumlah Pembelian dengan Diskon

Ukuran titik menunjukkan jumlah transaksi pada kombinasi jumlah dan persentase diskon.

In [ ]:
hubungan = (
    data_jual.groupby(["jumlah", "diskon_persen"])
    .size()
    .reset_index(name="frekuensi")
)

plt.figure(figsize=(9, 5))
plt.scatter(
    hubungan["jumlah"],
    hubungan["diskon_persen"],
    s=hubungan["frekuensi"] * 0.75 + 25,
    alpha=0.6,
    color="#8A5AA6"
)
plt.title("Jumlah Pembelian dan Diskon — 2025")
plt.xlabel("Jumlah Barang Dibeli (unit)")
plt.ylabel("Diskon (%)")
plt.xticks(sorted(hubungan["jumlah"].unique()))
plt.yticks(sorted(hubungan["diskon_persen"].unique()))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "visualisasi_jumlah_dan_diskon_2025.png", dpi=160)
plt.show()

## Hasil Gabungan 1:1

Empat grafik hasil eksekusi di atas digabung menjadi satu gambar persegi. Seluruh isi grafik dipertahankan; jarak antarpanel dibuat terlihat.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Gabungkan empat file PNG yang dibuat oleh sel-sel visualisasi di atas.
file_grafik = [
    OUTPUT_DIR / "visualisasi_penjualan_bulanan_2025.png",
    OUTPUT_DIR / "visualisasi_10_produk_terlaris_2025.png",
    OUTPUT_DIR / "visualisasi_status_transaksi_2025.png",
    OUTPUT_DIR / "visualisasi_jumlah_dan_diskon_2025.png",
]

ukuran_kanvas = 1200
margin = 42
jarak = 28
ukuran_panel = (ukuran_kanvas - (2 * margin) - jarak) // 2
ukuran_gambar = (ukuran_panel - 36, ukuran_panel - 36)

kanvas = Image.new("RGB", (ukuran_kanvas, ukuran_kanvas), "#EEF1F5")
draw = ImageDraw.Draw(kanvas)

for i, path in enumerate(file_grafik):
    if not path.exists():
        raise FileNotFoundError(f"Grafik sumber belum ditemukan: {path}")

    kolom = i % 2
    baris = i // 2
    x = margin + kolom * (ukuran_panel + jarak)
    y = margin + baris * (ukuran_panel + jarak)

    draw.rounded_rectangle(
        (x, y, x + ukuran_panel, y + ukuran_panel),
        radius=18,
        fill="#FFFFFF",
        outline="#DCE2EA",
        width=2,
    )

    with Image.open(path) as sumber:
        grafik = sumber.convert("RGB")
        grafik.thumbnail(ukuran_gambar, Image.Resampling.LANCZOS)
        posisi = (
            x + (ukuran_panel - grafik.width) // 2,
            y + (ukuran_panel - grafik.height) // 2,
        )
        kanvas.paste(grafik, posisi)

folder_hasil = OUTPUT_DIR.parent
folder_hasil.mkdir(parents=True, exist_ok=True)
png_hasil = folder_hasil / "visualisasi_gabungan_2025.png"
kanvas.save(png_hasil, format="PNG", optimize=True)

# SVG menyimpan kanvas yang sama agar tampil tajam pada pembesaran di README.
fig, ax = plt.subplots(figsize=(12, 12), dpi=100)
ax.imshow(kanvas)
ax.axis("off")
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
svg_hasil = folder_hasil / "visualisasi_gabungan_2025.svg"
fig.savefig(svg_hasil, format="svg", facecolor="#EEF1F5", transparent=False)
plt.close(fig)

print("Gambar gabungan:", png_hasil)
print("Ukuran kanvas:", kanvas.size)
display(kanvas)